# Train apart, stitch by a DAG tensor: a CNN and a tiny LM

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/stitch_llm_cnn_dag)

A small CNN learns MNIST digits and a tiny character-level GPT learns sentences such as "three+two=five.", each on its own. One DAG edge then joins them: a connector turns the CNN's features into soft tokens that replace the first word, so the LM answers "[image of a 3]+two=" with "five". The whole training process is itself a tensor DAG (rules on levels, a max-plus critical path, one batched call per rule over the seeds), and it is compared with the same network trained end to end on the joint task alone, with the same compute (multiply-adds). The stitching edge is also written as a mixture-of-experts layer with a fixed router, and a learned router is put to the test.

**Vector form**

$$X^0 = G_{\mathrm{img}} \odot C\big(\mathrm{CNN}(\mathrm{image})\big) + G_{\mathrm{txt}} \odot \mathrm{Tok}[\mathrm{text}],\qquad z = \mathrm{LM}(X^0 + P),\qquad \ell_r = \max_{s} (A_{\mathrm{rule}})_{rs}(\ell_s + 1),\qquad t = d \oplus (A_{\mathrm{rule}} \otimes t)$$

**Index (tensor) form**

$$q_{mne} = \mathrm{softmax}_e\,\mathrm{CNN}_m(x_{mn}) \qquad c_{mntc} = \mathrm{GELU}(q_{mne}W^1_{mef} + S_{mtf})\,W^2_{mfc} + b^2_{mc} \qquad X^0_{mntc} = G_{mntx}\,Y^x_{mntc} \qquad \mathcal L_m = -\frac{1}{|\mu_m|}\sum_{n,t}\mu_{mnt}\log \mathrm{softmax}_v z_{mntv}[y_{mnt}]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{\mathrm{rule}}$ | `rule_graph(parse_snakefile(training.smk))` | (R, R) | rule $s$ reads what rule $r$ writes; the training process parsed from data/training.smk with the Circle of Willis problem's parser |
| $A = \sum A_{\mathrm{rule}} \otimes M$ | `dag.job_adjacency(A_rule, axes, sizes)` | (J, J) | job graph over the wildcards seed and $n$: a rule without $n$ broadcasts to every $n$ ($I \otimes \mathbb 1$), evaluate gathers all ($\mathbb 1^\top$) |
| $\ell_r$ | `dag.levels(A_rule)` | (R,) | level of each rule: train_cnn, train_lm and the monolith on 0, the connector on 1, fine-tune and router on 2, evaluate on 3 |
| $t = d \oplus (A \otimes t)$ | `dag.finish_times(A_rule, seconds)` | (R,) | max-plus finish times from the measured rule durations; $\max t$ is the critical path |
| $q_{mne}$ | `cnn_output(x, w)` | (M, N, 10) | the CNN's ten digit probabilities for each image; the model axis $m$ becomes the groups of one grouped convolution per layer |
| $c_{mntc}$ | `connector(q, slots, w)` | (M, P, d) | the DAG edge: digit probabilities plus a slot vector $S_t$ → an LM input vector at position $t$; the $K = 5$ soft tokens replace the hidden first word |
| $G_{mntx}$ | `route_fixed(...), route_learned(r, w)` | (M, N, T, 2) | router: one-hot by data type (image slot or letter), or a learned Switch top-1 softmax over the position features |
| $X^0 = G_{mntx}Y^x_{mntc}$ | `einsum('mntx,xmntc->mntc', G, Y)` | (M, N, T, d) | repeated $x$ ⇒ summed: the combine of the MoE layer; each expert $Y^x$ is computed only on its dispatched group |
| $z_{mntv}$ | `lm_logits(X0, w)` | (M, N, T, V) | next-letter logits of the tiny GPT (einsum attention with a model axis, from chain_of_thought) |
| $\mathcal L_m$ | `masked_ce(z, y, mask)` | (M,) | per-model loss on the text after the prefix (nats); summing the $M$ losses gives every model its own gradient |

Idea. Networks can be trained separately, then connected through a DAG tensor, then optimised globally, and the whole process can itself be controlled by the DAG tensor. The idea is Jacky Ka Long Ko's; this problem tests it on a small CNN and a tiny character-level LM, deliberately not as a vision-language model trained jointly from scratch, and compares it with exactly that non-DAG alternative. What is new and what is not. Building a multimodal model from a separately pretrained image encoder, a pretrained LM and a small learned connector is how LLaVA (Liu et al. 2023, a linear or MLP projection into the LM's token space) and BLIP-2 (Li et al. 2023, the Q-Former between a frozen image encoder and a frozen LLM) are made; Frozen (Tsimpoukelli et al. 2021) feeds an image to a frozen LM as a soft prefix, as here; and model stitching (Lenc and Vedaldi, CVPR 2015, arXiv 2014; Bansal et al. 2021) joins two trained networks with a small trainable layer. What this problem adds is the orchestration: the model graph and the training process are both run by the tensor DAG engine of the Circle of Willis workflow problem (rule graph, Kronecker job graph, boolean levels, max-plus critical path), and the DAG-built model is measured against a monolith. Task and models. A CNN (two 5×5 convolutions, 8474 weights) learns the ten digits from 3000 MNIST images. A two-layer GPT (d = 32, four heads, 27 543 weights, einsum attention with a model axis from chain_of_thought) learns the 300 sentences "a op b = r." with op in +, −, × mod 10, the digits written as words ("three+two=five.", the first word right-aligned in a 5-letter slot). The joint task gives an MNIST test image of a and the text "+two="; the operand word is not in the text. The DAG edge is a connector (3808 weights): from the CNN's ten digit probabilities $q$ and a slot vector $S_t$ it makes $K = 5$ soft tokens $c_t = \mathrm{GELU}(qW_1 + S_t)W_2 + b_2$ that replace the operand slot at the LM input. (Feeding the CNN's penultimate features instead learned much more slowly in trials: about 0.25 joint accuracy after 100 connector steps against about 0.6.) Accuracy is the share of test examples whose answer letters and full stop are all the LM's top choice given the true prefix, which is exactly what greedy decoding writes. The training process as a DAG. data/training.smk lists the rules; the engine parses it with the Circle of Willis parser and runs it level by level, each rule as ONE batched call over its models on a stacked model axis (seed × joint-set size $n$): level 0 train_cnn ∥ train_lm ∥ train_monolith; level 1 train_connector (CNN and LM frozen, broadcast from seed to (seed, $n$) by the $I \otimes \mathbb 1$ block); level 2 global_finetune (everything, end to end through the edge, at a 100 times smaller learning rate) ∥ two learned routers; level 3 evaluate. With 2 seeds and 4 sizes: 8 rules, 45 jobs, 84 job edges, $A$ nilpotent. The tests check the levels, that a cycle is refused, that two seeds trained in one stack give the same losses as each trained alone (Adam is elementwise, clipping is per model) and that one model's loss sends zero gradient to the others. The baseline. The monolith is the same CNN + connector + LM (39 825 weights), randomly initialised and trained end to end on the joint examples only. Its budget is matched in multiply-adds per model: the DAG pipeline's 150 CNN, 600 LM, 150 connector and 80 fine-tune steps (batch 32) cost as much as 682 joint steps, which the monolith gets (counting each pretraining in full for every model, although in the DAG one pretrained CNN and LM serve all four values of $n$). Results (2 seeds, 1000 test images, chance 0.1, CPU). Multithreaded float reductions make repeated runs differ by a few points, so the figures below are the range of the seed-mean over four full runs; results/stages.csv holds the exact numbers of the bundled run. Alone, the CNN reads 0.89 of the test digits and the LM completes 0.97 to 0.99 of the text prompts; writing the CNN's digit as text and giving it to the LM scores 0.88 to 0.89, the ceiling of this pipeline. Joint accuracy for $n$ = 100, 300, 1000, 3000 joint examples: untrained connector about 0.16; after the connector stage 0.41–0.47, 0.58–0.63, 0.60–0.69, 0.64–0.68; after the global fine-tune 0.41–0.47, 0.63–0.67, 0.68–0.75, 0.71–0.75; the monolith 0.14, 0.13–0.15, 0.16–0.17, 0.23. The DAG-built model wins at every $n$ in every run, by the most where joint data are scarce, and the fine-tune adds 0.03 to 0.08 from $n$ = 300 on (at $n$ = 100 it changes nothing on average). The reason is plain: the DAG reuses unimodal data (3000 labelled digits and every sentence), while the monolith must discover digits and arithmetic from $n$ answer words alone, and × often hides the operand (anything × 0 = 0). With more steps or data the monolith would improve (its loss is still falling at the end of its budget): this is a small-budget comparison, not a law. DAG edge = MoE dispatch. The same input layer is written as a mixture of experts: expert img (CNN + connector) and expert txt (the LM's token embedding), a router $G$ over positions × experts, each expert run batched on the positions dispatched to it, and the combine $X^0 = G_{mntx}Y^x_{mntc}$. With $G$ one-hot by data type (image slot or letter, known before any data is seen, as a workflow picks a rule by its wildcard) the MoE output equals the DAG-wired output to 1e-6, and so do all gradients (tested). The precise relation: a DAG edge is an MoE dispatch with a fixed router decided by the data type; an MoE is a DAG whose edges are chosen per token by a learned router. Can a router learn that routing? A linear router on each position's features (7×7 pooled image and one-hot id) was trained with the experts frozen and the Switch load-balancing loss ($\alpha$ = 0.01). Switch top-1 (only the chosen expert runs, scaled by $p$) is unreliable: its routing accuracy ranged from 0 to 0.92 over models and runs, and its task accuracy from 0 to about 0.46, because the unchosen expert gets no gradient, so a wrong early choice is never corrected, and the balance loss pulls towards a 50/50 split while the true split is 5/17. Trained as a dense mixture ($G = p$, both experts everywhere) and then used top-1, the router recovers the modality routing in every model and run (0.97 to 0.99 of positions) and reaches about 0.35 to 0.65 task accuracy, a little below the fixed router because $p < 1$ scales the outputs. When the right edge is known from the data type, fixing it is both exact and cheaper. Time. On a 6-core desktop CPU the default run trains in about 66 s when the machine is quiet (up to 2 to 4 minutes when it is shared with other heavy jobs), plus about 15 s for the GIF: in the quiet run the monolith took 35 s, the LM 12 s, the connector 4.5 s and the fine-tune 4 s. With one worker per rule the max-plus critical path is the monolith, then evaluate (39 s); the DAG system's own chain LM → connector → fine-tune → evaluate takes about 24 s. The code picks the GPU when torch sees one; this project's environment has a CPU build of torch, so every bundled number is from the CPU. Data. MNIST (Y. LeCun, C. Cortes, C. J. C. Burges), licence CC BY-SA 3.0: a balanced subset of 3000 training and 1000 test images is bundled (data/mnist_subset.npz, 0.7 MB; data/SOURCES.md has the source, the checksums and the processing). The sentences are generated in code. Try. Fewer LM steps (the LM's accuracy caps everything), a larger $\alpha$ (the Switch router collapses more), no fine-tune (0 steps), or one seed to see how much the seeds disagree.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/stitch_llm_cnn_dag/main.py
"""Main loop: parse the training rule graph → run it with the tensor DAG engine (each rule one batched call over the
models: seeds × joint-set sizes) → accuracy of every stage vs the monolith, sample efficiency, fixed vs learned
router, measured schedule → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation as eq
from . import plot, solve
from . import tensors as T


def _pm(x: torch.Tensor) -> str:
    """'mean ± half-range' over the seeds."""
    return f"{x.mean():.3f} ± {(x.max() - x.min()) / 2:.3f}"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    f = 0.03 if fast else 1.0
    cfg = solve.Config(
        seeds=list(range(1 if fast else int(params["seeds"]))),
        sizes=[100, 3000] if fast else [100, 300, 1000, 3000],
        steps_cnn=max(2, int(params["steps_cnn"] * f)),
        steps_lm=max(2, int(params["steps_lm"] * f)),
        steps_con=max(2, int(params["steps_con"] * f)),
        steps_ft=max(2, int(params["steps_ft"] * f)),
        steps_router=max(2, int(params["steps_router"] * f)),
        lr_ft=params["lr_ft"],
        alpha=params["alpha"],
        n_test=100 if fast else 1000,
    )
    S, N = len(cfg.seeds), len(cfg.sizes)
    d = solve.build_dag(cfg)
    lev = dict(zip(d["names"], d["levels"].tolist()))
    print(f"training DAG ({T.TRAINING_GRAPH.name}): {len(d['names'])} rules, {len(d['A'])} jobs, "
          f"{int(d['A'].sum())} job edges; A nilpotent: {solve.dag.is_nilpotent(d['A'])}")  # fmt: skip
    for k in range(int(d["levels"].max()) + 1):
        print(f"  level {k}: " + " ∥ ".join(n for n, lv in lev.items() if lv == k))
    t0 = time.perf_counter()
    st, seconds, d = solve.run_engine(cfg, dev, d)
    total = time.perf_counter() - t0
    sch = solve.schedule(d, seconds)
    models = {"train_cnn": S, "train_lm": S, "evaluate": 1}  # models in each rule's one batched call
    models.update({n: S * N for n in d["names"] if n not in models})
    print(f"device {dev}; {S} seeds × {N} joint-set sizes {cfg.sizes}; batch {cfg.batch}")
    for n in d["names"]:
        print(f"  {n:16s} level {lev[n]}  {models[n]:2d} models in one batched call  {seconds[n]:6.1f} s")
    print(f"serial (one worker, level by level) {sch['serial']:.1f} s; with one worker per rule the max-plus "
          f"critical path is {' → '.join(sch['path'])} = {sch['makespan']:.1f} s")  # fmt: skip

    ev = st["evaluate"]
    mono_count = sum(T.count(st["train_monolith"][k]) for k in ("cnn", "con", "lm"))
    print(f"weights per model: CNN {T.count(st['train_cnn']['w'])}, connector "
          f"{T.count(st['train_connector']['con'])}, LM {T.count(st['train_lm']['w'])}; monolith {mono_count}"
          " (the same architecture)")  # fmt: skip
    print(f"budget: the DAG pipeline takes {cfg.steps_cnn} + {cfg.steps_lm} + {cfg.steps_con} + {cfg.steps_ft} "
          f"optimiser steps of {cfg.batch}; the monolith gets the same {cfg.steps_mono} steps of {cfg.batch}, "
          "all on the joint task")  # fmt: skip
    print(f"CNN alone, MNIST test digits: {_pm(ev['cnn_acc'])};  LM alone, all 300 text prompts: {_pm(ev['lm_acc'])};"
          f"  CNN's label written as text → LM: {_pm(ev['symbolic'])}")  # fmt: skip
    print("joint test accuracy (1000 images + prompts, chance 0.1), mean ± half-range over seeds:")
    print("  n     " + "".join(f"{k:>18s}" for k in plot.STAGE))
    for j, n in enumerate(cfg.sizes):
        print(f"  {n:<5d} " + "".join(f"{_pm(ev['acc'][k][:, j]):>18s}" for k in plot.STAGE))
    for k, label in (("router", "Switch top-1"), ("router_soft", "trained dense")):
        print(
            f"learned router ({label}), share of positions sent to the modality's expert: "
            + ", ".join(f"n={n}: {_pm(ev['route_acc'][k][:, j])}" for j, n in enumerate(cfg.sizes))
        )

    # fixed vs learned router on a few test examples (seed 0, largest n), and test completions
    m = N - 1  # model (seed 0, largest n)
    data, test, c = st["data"], ev["test"], st["train_connector"]
    rows = torch.arange(12, device=dev)
    bt = solve.joint_batch(data, rows[None], test["op"][rows][None], test["b"][rows][None], train=False)
    G_learned = {}
    with torch.no_grad():
        for k in ("router", "router_soft"):
            _, G, p = solve.moe_forward(bt, *(T.select(c[j], [m]) for j in ("cnn", "con", "lm")),
                                        T.select(st["train_" + k]["router"], [m]), data["blank"])  # fmt: skip
            G_learned[k] = p[0, :, :, 0].cpu().numpy()
    ids, is_img = eq.position_items(bt["inp"])
    G_fixed = eq.route_fixed(1, len(rows), is_img)[0, :, :, 0]
    ex_rows = torch.arange(4, device=dev)
    ft, mono = st["global_finetune"], st["train_monolith"]
    tok = data["table"]["tokens"][data["y_test"][ex_rows], test["op"][ex_rows], test["b"][ex_rows]]
    x = data["x_test"][ex_rows]
    dag_out = solve.greedy(*(T.select(ft[k], [m]) for k in ("cnn", "con", "lm")), x, tok)
    mono_out = solve.greedy(*(T.select(mono[k], [m]) for k in ("cnn", "con", "lm")), x, tok)
    examples = []
    for i in range(len(ex_rows)):
        s = T.decode(tok[i])
        examples.append({"img": x[i].cpu().numpy(), "prompt": s[T.K : s.index("=") + 1], "dag": dag_out[i],
                         "mono": mono_out[i], "truth": s[s.index("=") + 1 : s.index(".")]})  # fmt: skip
        print(f"  test image of a {T.WORDS[int(data['y_test'][i])]}, prompt '{examples[-1]['prompt']}' → DAG "
              f"'{dag_out[i]}', monolith '{mono_out[i]}' (true '{examples[-1]['truth']}')")  # fmt: skip
    print(f"total {total:.0f} s")

    res = {"names": d["names"], "levels": d["levels"].tolist(), "A_rule": d["A_rule"].tolist(), "seconds": seconds,
           "path": sch["path"], "finish": sch["finish"].tolist(), "makespan": sch["makespan"], "serial": sch["serial"],
           "models": models, "loss": {n: st[n]["loss"] for n in d["names"] if n != "evaluate"},
           "acc": ev["acc"], "route_acc": ev["route_acc"], "cnn_acc": ev["cnn_acc"], "lm_acc": ev["lm_acc"],
           "symbolic": ev["symbolic"], "sizes": cfg.sizes, "seeds": cfg.seeds, "G_fixed": G_fixed.cpu().numpy(),
           "G_learned": G_learned, "route_acc_show": {k: float(v[0, -1]) for k, v in ev["route_acc"].items()},
           "examples": examples}  # fmt: skip
    plot.write_tables(res, out_dir)
    plot.schedule_png(res, out_dir / "schedule.png")
    return plot.animate(res, out_dir / "stitch_llm_cnn_dag.gif", n_frames=4 if fast else 36, hold=2 if fast else 10)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/stitch_llm_cnn_dag/equation.py
r"""Two separately trained networks stitched by one DAG edge, and the same edge written as a mixture-of-experts layer.

Indices: m = model on the stacked axis (seed, joint-set size), n = example, t = input position, c = LM channel (d),
e = digit class of the CNN output, f = hidden channel, v = vocabulary letter, x = expert (img, txt).

    q_{mne} = \mathrm{softmax}_e\,\mathrm{CNN}_m(\mathrm{image}_{mn})          (M, N, 10) digit probabilities
    c_{mntc} = \mathrm{GELU}(q_{mne} W^1_{mef} + S_{mtf}) W^2_{mfc} + b^2_{mc}     (e, f summed) connector, slot t
    X^0_{mntc} = c_{mntc} for t < K,   \mathrm{Tok}_{m, t_{mnt}, c} for t ≥ K       the DAG edge: soft prefix
    z_{mntv} = \mathrm{LM}_m(X^0 + P)                                          next-letter logits

As an MoE layer at the LM input (x = img, txt; G the router, one-hot):

    Y^{img}_{mntc} = c_{mntc},  Y^{txt}_{mntc} = \mathrm{Tok}_{m, t_{mnt}, c}       each expert run on its group only
    X^0_{mntc} = G_{mntx} Y^x_{mntc}                                           (x summed) the combine
    fixed router:   G_{mnt,img} = [t < K]        (known from the data type, like a workflow rule chosen by wildcard)
    learned router: p = \mathrm{softmax}(r_{mnt} W^R + b^R),  G = p ⊙ \mathrm{onehot}(\arg\max p)   (Switch top-1)
                    or trained with the dense G = p (both experts everywhere), then used top-1
    balance loss:   \mathcal L_{bal} = X \sum_x f_x P_x,   f_x = share of positions sent to x, P_x = mean p_x

The LM blocks (pre-norm attention and MLP with a model axis) are imported from problems/ai/chain_of_thought.
"""

import torch
import torch.nn.functional as F

from problems.ai.chain_of_thought.equation import layer_norm, mlp, self_attention

from . import tensors as T


# ---------------------------------------------------------------- the CNN (one grouped convolution per layer)
def cnn_features(x: torch.Tensor, w: dict) -> torch.Tensor:
    """Images x (M, N, 28, 28), model m sees its own images → h (M, N, E). The model axis becomes conv groups:
    channel block m of every layer only ever meets model m's filters, so the M networks never mix."""
    m, n = x.shape[:2]
    z = x.transpose(0, 1)  # (N, M, 28, 28): one input channel per model
    z = F.max_pool2d(F.relu(F.conv2d(z, w["c1"].flatten(0, 1), w["b1"].flatten(), groups=m)), 2)  # (N, 6M, 12, 12)
    z = F.max_pool2d(F.relu(F.conv2d(z, w["c2"].flatten(0, 1), w["b2"].flatten(), groups=m)), 2)  # (N, 12M, 4, 4)
    z = z.reshape(n, m, 192).transpose(0, 1)  # (M, N, 192)
    return F.relu(torch.einsum("mnk,mke->mne", z, w["fc"]) + w["bf"][:, None])


def cnn_logits(h: torch.Tensor, w: dict) -> torch.Tensor:
    """Digit logits (M, N, 10) from the features h (M, N, E)."""
    return torch.einsum("mne,mek->mnk", h, w["head"]) + w["bh"][:, None]


def cnn_output(x: torch.Tensor, w: dict) -> torch.Tensor:
    """q = softmax(CNN logits): the 10 digit probabilities, what the CNN passes along the DAG edge.  → (M, N, 10)"""
    return torch.softmax(cnn_logits(cnn_features(x, w), w), -1)


# ---------------------------------------------------------------- the LM on input vectors
def lm_logits(X0: torch.Tensor, w: dict) -> torch.Tensor:
    """Input vectors X0 (M, N, T, d) (token embeddings or soft tokens) → next-letter logits (M, N, T, V)."""
    X = X0 + w["pos"][:, None, : X0.shape[2]]
    for p in w["layers"]:
        X = X + self_attention(layer_norm(X, p["g1"], p["b1"]), p)[0]
        X = X + mlp(layer_norm(X, p["g2"], p["b2"]), p)
    return torch.einsum("mntc,mcv->mntv", layer_norm(X, w["gf"], w["bf"]), w["head"]) + w["bias"][:, None, None]


def embed(tokens: torch.Tensor, w: dict) -> torch.Tensor:
    """Tok_{m, t_{mnt}, c}: model m looks up its own embedding table.  tokens (M, N, T) → (M, N, T, d)."""
    model = torch.arange(tokens.shape[0], device=tokens.device)[:, None, None]
    return w["tok"][model, tokens]


# ---------------------------------------------------------------- the connector and the DAG-wired model
def connector(q: torch.Tensor, slots: torch.Tensor, w: dict) -> torch.Tensor:
    """c_t(q) = GELU(q W1 + S_t) W2 + b2.  q (M, P, 10) CNN outputs, slots (M, P) positions t → (M, P, d)."""
    model = torch.arange(q.shape[0], device=q.device)[:, None]
    pre = torch.einsum("mpe,mef->mpf", q, w["W1"]) + w["S"][model, slots]
    return torch.einsum("mpf,mfc->mpc", F.gelu(pre), w["W2"]) + w["b2"][:, None]


def stitched_inputs(x: torch.Tensor, tokens: torch.Tensor, w_cnn: dict, w_con: dict, w_lm: dict) -> torch.Tensor:
    """The DAG edge CNN → connector → LM: K soft tokens from the image replace the operand word.
    x (M, N, 28, 28), tokens (M, N, T) → X0 (M, N, T, d)."""
    m, n, t = tokens.shape
    h = cnn_output(x, w_cnn)  # (M, N, 10)
    hk = h[:, :, None].expand(m, n, T.K, h.shape[-1]).reshape(m, n * T.K, -1)
    slots = torch.arange(T.K, device=x.device).repeat(n)[None].expand(m, -1)  # (M, N·K) slot of every row
    prefix = connector(hk, slots, w_con).reshape(m, n, T.K, -1)
    return torch.cat([prefix, embed(tokens[:, :, T.K :], w_lm)], dim=2)


def stitched_logits(x, tokens, w_cnn, w_con, w_lm) -> torch.Tensor:
    return lm_logits(stitched_inputs(x, tokens, w_cnn, w_con, w_lm), w_lm)


# ---------------------------------------------------------------- the same edge as an MoE layer
def position_items(tokens: torch.Tensor):
    """What sits at each input position: an id (IMG placeholder at the K image positions, else the letter) and an
    image (the digit at image positions, a blank page at text positions).  → ids (M, N, T), is_img (T,) bool."""
    t = tokens.shape[2]
    is_img = torch.arange(t, device=tokens.device) < T.K
    ids = torch.where(is_img, torch.full_like(tokens, T.STOI[T.IMG]), tokens)
    return ids, is_img


def router_features(x: torch.Tensor, ids: torch.Tensor, is_img: torch.Tensor, blank: torch.Tensor) -> torch.Tensor:
    """r_{mnt} = 7×7 average-pooled image of the position ⊕ one-hot id.  → (M, N, T, 49 + V)."""
    m, n, t = ids.shape
    pooled = F.avg_pool2d(x.reshape(m * n, 1, 28, 28), 4).reshape(m, n, 1, 49)
    pooled_blank = F.avg_pool2d(blank[None, None], 4).reshape(1, 1, 1, 49)
    img = torch.where(is_img[None, None, :, None], pooled, pooled_blank)  # (M, N, T, 49)
    return torch.cat([img, F.one_hot(ids, len(T.VOCAB)).to(x.dtype)], dim=-1)


def route_fixed(m: int, n: int, is_img: torch.Tensor) -> torch.Tensor:
    """G (M, N, T, 2) one-hot by modality: expert 0 = img at the K prefix positions, expert 1 = txt elsewhere."""
    g = torch.stack([is_img, ~is_img], -1).to(torch.get_default_dtype())
    return g[None, None].expand(m, n, -1, -1)


def route_learned(r: torch.Tensor, w: dict, soft: bool = False) -> tuple[torch.Tensor, torch.Tensor]:
    """Switch top-1: p = softmax(r W + b), G = p ⊙ onehot(argmax p).  r (M, N, T, F) → G, p (M, N, T, 2).
    soft=True: G = p, a dense mixture (both experts run everywhere), used only while training the soft router."""
    p = torch.softmax(torch.einsum("mntf,mfx->mntx", r, w["W"]) + w["b"][:, None, None], -1)
    return (p if soft else p * F.one_hot(p.argmax(-1), 2).to(p.dtype)), p


def balance_loss(G: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Switch load-balancing loss X Σ_x f_x P_x per model (M,): 1 when the split is even, X·max when one expert
    takes all."""
    f = (G > 0).to(p.dtype).mean((1, 2))  # (M, X) share of positions dispatched to each expert
    return p.shape[-1] * (f * p.mean((1, 2))).sum(-1)


def dispatch(mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """The positions of one expert's group, per model, packed to the front: mask (M, S) → idx (M, P), valid (M, P).
    P = the largest group over the models; shorter groups are padded (valid = False)."""
    cnt = mask.sum(1)
    P = int(cnt.max()) if mask.numel() else 0
    idx = torch.argsort((~mask).to(torch.uint8), dim=1, stable=True)[:, :P]
    return idx, torch.arange(P, device=mask.device)[None] < cnt[:, None]


def moe_inputs(x, ids, G, w_cnn, w_con, w_lm, blank) -> torch.Tensor:
    """X0_{mntc} = G_{mntx} Y^x_{mntc}: dispatch every position to its expert, run each expert batched on its own
    group (CNN features once per image, connector per routed position; token lookup), scatter back, combine."""
    m, n, t = ids.shape
    flat_ids = ids.reshape(m, n * t)
    h_img = cnn_output(x, w_cnn)  # (M, N, 10): the CNN runs once per image ...
    h_blank = cnn_output(blank.expand(m, 1, 28, 28), w_cnn)  # (M, 1, 10) ... and once on the blank page
    is_img = torch.arange(t, device=x.device) < T.K
    Y = []
    for x_e in range(2):  # the two experts
        idx, valid = dispatch(G[..., x_e].reshape(m, n * t) > 0)
        if x_e == 0:  # E_img = CNN + connector at position t of example n
            ex, slot = idx // t, idx % t
            h = torch.where(is_img[slot][..., None], h_img.gather(1, ex[..., None].expand(-1, -1, h_img.shape[-1])),
                            h_blank)  # fmt: skip
            out = connector(h, slot, w_con)
        else:  # E_txt = the LM's own token embedding
            model = torch.arange(m, device=x.device)[:, None]
            out = w_lm["tok"][model, flat_ids.gather(1, idx)]
        out = out * valid[..., None]
        Y.append(torch.zeros(m, n * t, out.shape[-1], dtype=out.dtype, device=x.device).scatter_add(
            1, idx[..., None].expand_as(out), out).reshape(m, n, t, -1))  # fmt: skip
    return torch.einsum("mntx,xmntc->mntc", G, torch.stack(Y))


# ---------------------------------------------------------------- losses and accuracy
def masked_ce(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Per-model mean −log p of the masked targets (nats).  logits (M, N, T, V), targets, mask (M, N, T) → (M,)."""
    nll = -torch.gather(torch.log_softmax(logits, -1), -1, targets[..., None])[..., 0]
    return (nll * mask).sum((1, 2)) / mask.sum((1, 2))


def exact(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Every masked letter is the argmax given the true prefix ⇔ greedy decoding writes exactly these letters.
    → (M, N) bool."""
    return ((logits.argmax(-1) == targets) | ~mask).all(-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/stitch_llm_cnn_dag/tensors.py
"""Build the concrete tensors: the MNIST subset, the synthetic sentences, the joint task, and stacked weights.

data/mnist_subset.npz   3000 training and 1000 test MNIST digits (300 and 100 per class), uint8, see data/SOURCES.md
data/training.smk       the training process as a Snakefile-style rule graph, run by the tensor DAG engine

Every weight tensor has the model axis m first: model m = (seed, joint-set size) row-major, so one optimiser step
trains every seed (and every size) at once, each model only touching its own slice.
"""

import math
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data"
TRAINING_GRAPH = DATA / "training.smk"

WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
OPS = ["plus", "minus", "times"]  # mod 10
SYM = "+-*"  # how the text writes them
K = 5  # the first operand word is right-aligned in a 5-character slot: the K soft prefix tokens replace it
CTX = 18  # "seven*seven=seven." is 5 + 13 characters
PAD, IMG = "_", "#"  # padding after the full stop; the placeholder id of an image position (never in the text)
VOCAB = PAD + IMG + " .=" + SYM + "".join(sorted({c for w in WORDS for c in w}))
STOI = {c: i for i, c in enumerate(VOCAB)}
MEAN, STD = 0.1307, 0.3081  # MNIST pixel normalisation


def result(a: torch.Tensor, op: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """r = a + b, a − b or a × b (mod 10) for op = 0, 1, 2.  Any shape (broadcast)."""
    return torch.where(op == 0, a + b, torch.where(op == 1, a - b, a * b)) % 10


def sentence(a: int, op: int, b: int) -> str:
    """'  two+three=five.' padded with '_' to CTX characters."""
    r = int(result(torch.tensor(a), torch.tensor(op), torch.tensor(b)))
    return f"{WORDS[a]:>{K}}{SYM[op]}{WORDS[b]}={WORDS[r]}.".ljust(CTX, PAD)


def encode(s: str) -> list[int]:
    return [STOI[c] for c in s]


def decode(ids) -> str:
    return "".join(VOCAB[int(i)] for i in ids)


def sentence_table() -> dict:
    """Every sentence of the language at once, indexed by (a, op, b): 10 × 3 × 10 = 300.

    tokens (10, 3, 10, CTX) ids;  text_mask (…, CTX − 1) targets up to and including the full stop;  answer_mask
    (…, CTX − 1) targets of the result word and its full stop: getting all of them right under teacher forcing is
    exactly what greedy decoding writes, so accuracy needs one forward pass.
    """
    tok = torch.zeros(10, 3, 10, CTX, dtype=torch.long)
    text = torch.zeros(10, 3, 10, CTX - 1, dtype=torch.bool)
    ans = torch.zeros(10, 3, 10, CTX - 1, dtype=torch.bool)
    for a in range(10):
        for op in range(3):
            for b in range(10):
                s = sentence(a, op, b)
                tok[a, op, b] = torch.tensor(encode(s))
                stop = s.index(".")
                start = s.index("=") + 1  # first character of the result word
                text[a, op, b, :stop] = True  # target j is character j + 1
                ans[a, op, b, start - 1 : stop] = True
    return {"tokens": tok, "text_mask": text, "answer_mask": ans}


def load_mnist(device: torch.device) -> dict:
    """The bundled subset as normalised float images (N, 28, 28) and integer labels (N,)."""
    with np.load(DATA / "mnist_subset.npz") as z:  # images (N, 28, 28) uint8, labels (N,), split 0 train / 1 test
        images, labels, split = z["images"], z["labels"], z["split"]
    arr = {"x_train": images[split == 0], "y_train": labels[split == 0], "x_test": images[split == 1],
           "y_test": labels[split == 1]}  # fmt: skip

    def img(x):
        return ((torch.from_numpy(x).float() / 255 - MEAN) / STD).to(device)

    return {"x_train": img(arr["x_train"]), "y_train": torch.from_numpy(arr["y_train"]).long().to(device),
            "x_test": img(arr["x_test"]), "y_test": torch.from_numpy(arr["y_test"]).long().to(device),
            "blank": torch.full((28, 28), -MEAN / STD, device=device)}  # fmt: skip


def joint_pool(seeds: list[int], n_images: int, device: torch.device) -> dict:
    """Joint training examples per seed: a random order of the training images, each with a random (op, b).
    A set of size n is the first n rows, so the sets are nested (100 ⊂ 300 ⊂ …).  (S, n_images) each."""
    out = {"img": [], "op": [], "b": []}
    for s in seeds:
        g = torch.Generator().manual_seed(10_000 + s)
        out["img"].append(torch.randperm(n_images, generator=g))
        out["op"].append(torch.randint(3, (n_images,), generator=g))
        out["b"].append(torch.randint(10, (n_images,), generator=g))
    return {k: torch.stack(v).to(device) for k, v in out.items()}


def joint_test(n: int, device: torch.device) -> dict:
    """The fixed joint test set: test image i with its own random (op, b), the same for every model."""
    g = torch.Generator().manual_seed(4242)
    return {"op": torch.randint(3, (n,), generator=g).to(device), "b": torch.randint(10, (n,), generator=g).to(device)}


def draws(seeds: list[int], n_per_model: list[int], steps: int, batch: int, salt: int, device) -> torch.Tensor:
    """Minibatch row indices (steps, M, B): model m draws from range(n_m) with its own generator, so a model's
    batches never depend on which other models share the stack (batched run = per-model runs)."""
    out = [torch.randint(n, (steps, batch), generator=torch.Generator().manual_seed(s * 7919 + salt))
           for s, n in zip(seeds, n_per_model)]  # fmt: skip
    return torch.stack(out, 1).to(device)


# ---------------------------------------------------------------- stacked weights (model axis first)
def _stack(seeds: list[int], salt: int, make, device) -> dict:
    """make(gen) → dict of tensors for ONE model; stacked over the seeds, each drawn from its own generator."""
    ones = [make(torch.Generator().manual_seed(s * 100 + salt)) for s in seeds]

    def stack(xs):
        if isinstance(xs[0], dict):
            return {k: stack([x[k] for x in xs]) for k in xs[0]}
        if isinstance(xs[0], list):
            return [stack([x[i] for x in xs]) for i in range(len(xs[0]))]
        return torch.stack(xs).to(device).requires_grad_(True)

    return stack(ones)


def init_cnn(seeds: list[int], salt: int, device, emb: int = 32) -> dict:
    """conv 5×5 1→6, pool, conv 5×5 6→12, pool, 192 → emb features, emb → 10 digit logits."""

    def make(g):
        def u(*shape, fan):
            return (torch.rand(*shape, generator=g) * 2 - 1) * math.sqrt(6 / fan)  # He-uniform

        return {"c1": u(6, 1, 5, 5, fan=25), "b1": torch.zeros(6), "c2": u(12, 6, 5, 5, fan=150),
                "b2": torch.zeros(12), "fc": u(192, emb, fan=192), "bf": torch.zeros(emb),
                "head": u(emb, 10, fan=emb) / 2, "bh": torch.zeros(10)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_lm(seeds: list[int], salt: int, device, d: int = 32, heads: int = 4, layers: int = 2) -> dict:
    """The tiny GPT of chain_of_thought / tiny_transformer, one per model: tok (M, V, d), pos (M, CTX, d), blocks."""
    v, dk, s = len(VOCAB), d // heads, 1 / math.sqrt(d)

    def make(g):
        def r(*shape, scale):
            return scale * torch.randn(*shape, generator=g)

        def layer():
            return {"g1": torch.ones(d), "b1": torch.zeros(d), "Wq": r(heads, d, dk, scale=s),
                    "bq": torch.zeros(heads, dk), "Wk": r(heads, d, dk, scale=s), "bk": torch.zeros(heads, dk),
                    "Wv": r(heads, d, dk, scale=s), "bv": torch.zeros(heads, dk),
                    "Wo": r(heads, dk, d, scale=s / math.sqrt(2 * layers)), "bo": torch.zeros(d),
                    "g2": torch.ones(d), "b2": torch.zeros(d), "W1": r(d, 4 * d, scale=s), "c1": torch.zeros(4 * d),
                    "W2": r(4 * d, d, scale=0.5 * s / math.sqrt(2 * layers)), "c2": torch.zeros(d)}  # fmt: skip

        return {"tok": r(v, d, scale=0.3), "pos": r(CTX, d, scale=0.3), "layers": [layer() for _ in range(layers)],
                "gf": torch.ones(d), "bf": torch.zeros(d), "head": r(d, v, scale=1e-3),
                "bias": torch.zeros(v)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_connector(seeds: list[int], salt: int, device, d: int = 32, emb: int = 10, hidden: int = 64) -> dict:
    """c_t(q) = W2ᵀ GELU(W1ᵀ q + S_t) + b2: the CNN output q (10 digit probabilities) and a slot vector S_t → an LM
    input vector."""

    def make(g):
        return {"W1": torch.randn(emb, hidden, generator=g) / math.sqrt(emb),
                "S": 0.1 * torch.randn(CTX - 1, hidden, generator=g),
                "W2": torch.randn(hidden, d, generator=g) * 0.3 / math.sqrt(hidden), "b2": torch.zeros(d)}  # fmt: skip

    return _stack(seeds, salt, make, device)


def init_router(seeds: list[int], salt: int, device) -> dict:
    """Linear router over the position features (7×7 pooled image ⊕ one-hot id) → 2 expert logits."""
    f = 49 + len(VOCAB)

    def make(g):
        return {"W": 0.01 * torch.randn(f, 2, generator=g), "b": torch.zeros(2)}

    return _stack(seeds, salt, make, device)


def leaves(w) -> list[torch.Tensor]:
    """Every tensor of a nested weight dict/list."""
    if isinstance(w, dict):
        return [t for v in w.values() for t in leaves(v)]
    if isinstance(w, list):
        return [t for v in w for t in leaves(v)]
    return [w]


def clone(w, grad: bool):
    """Detached copy of a nested weight dict; trainable (requires_grad) or frozen."""
    if isinstance(w, dict):
        return {k: clone(v, grad) for k, v in w.items()}
    if isinstance(w, list):
        return [clone(v, grad) for v in w]
    return w.detach().clone().requires_grad_(grad)


def expand(w, n: int):
    """Repeat each model n times along the model axis (seed → (seed, size)): the I ⊗ 𝟙 edge of the job DAG."""
    if isinstance(w, dict):
        return {k: expand(v, n) for k, v in w.items()}
    if isinstance(w, list):
        return [expand(v, n) for v in w]
    return w.detach().repeat_interleave(n, 0)


def select(w, idx: list[int]):
    """Models idx of a stack."""
    if isinstance(w, dict):
        return {k: select(v, idx) for k, v in w.items()}
    if isinstance(w, list):
        return [select(v, idx) for v in w]
    return w[idx]


def count(w) -> int:
    """Number of weights of ONE model of the stack."""
    return sum(t[0].numel() for t in leaves(w))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/stitch_llm_cnn_dag/solve.py
"""The training process as a tensor DAG: every rule of data/training.smk is ONE batched call over its models.

L0  train_cnn ∥ train_lm ∥ train_monolith   (separate pretraining; the monolith baseline is just another rule)
L1  train_connector                          (CNN and LM frozen, only the connector learns)
L2  global_finetune ∥ train_router ∥ train_router_soft   (all weights end to end; or a learned MoE router on the
                                                          frozen experts, Switch top-1 or trained dense)
L3  evaluate

The DAG algebra (Kronecker job graph, boolean levels, max-plus finish times, critical path) is imported from the
Circle of Willis workflow problem, problems/medical/cow_workflow_dag.
"""

import math
import re
import time
from dataclasses import dataclass, field

import torch

from problems._common.progress import steps as progress_steps
from problems.medical.cow_workflow_dag import equation as dag
from problems.medical.cow_workflow_dag.solve import parse_snakefile

from . import equation as eq
from . import tensors as T


@dataclass
class Config:
    seeds: list[int] = field(default_factory=lambda: [0, 1])
    sizes: list[int] = field(default_factory=lambda: [100, 300, 1000, 3000])  # joint training examples n
    batch: int = 32
    steps_cnn: int = 150
    steps_lm: int = 600
    steps_con: int = 150
    steps_ft: int = 80
    steps_router: int = 60
    d: int = 32
    heads: int = 4
    layers: int = 2
    lr: float = 3e-3  # CNN and router
    lr_lm: float = 1e-2
    lr_con: float = 3e-2
    lr_ft: float = 3e-4  # global fine-tune: a gentle nudge of everything
    lr_mono: float = 2e-3
    alpha: float = 0.01  # Switch load-balancing coefficient
    n_test: int = 1000

    def macs(self) -> dict:
        """Multiply-adds of one forward pass per example (backward ≈ 2× forward for every one, so ratios hold)."""
        d, t, v = self.d, T.CTX - 1, len(T.VOCAB)
        cnn = 6 * 24 * 24 * 25 + 12 * 8 * 8 * 150 + 192 * 32 + 32 * 10
        lm = t * (self.layers * (12 * d * d + 2 * t * d) + d * v)  # Q, K, V, O, MLP 4d; scores and mix; read-out
        con = T.K * (10 * 64 + 64 * d)
        return {"cnn": cnn, "lm": lm, "joint": cnn + con + lm}

    @property
    def steps_mono(self) -> int:
        """Matched compute: the monolith's joint steps (same batch) cost as many multiply-adds as the DAG pipeline's
        CNN, LM, connector and fine-tune steps together, all counted per model."""
        c = self.macs()
        work = self.steps_cnn * c["cnn"] + self.steps_lm * c["lm"] + (self.steps_con + self.steps_ft) * c["joint"]
        return round(work / c["joint"])


# ---------------------------------------------------------------- the rule graph (parsed with the CoW parser)
def rule_graph(rules: list[dict]) -> tuple[torch.Tensor, list[tuple]]:
    """A_rule (R, R): rule s reads a file pattern rule r writes; and each rule's wildcards (seed, n)."""
    A = torch.zeros(len(rules), len(rules))
    for s, rs in enumerate(rules):
        for pat, _ in rs["inputs"]:
            for r, rr in enumerate(rules):
                if pat in rr["outputs"]:
                    A[s, r] = 1
    axes = [tuple(a for a in ("seed", "n") if any("{" + a + "}" in p for p in r["outputs"])) for r in rules]
    return A, axes


def build_dag(cfg: Config, text: str | None = None) -> dict:
    """Rules, A_rule, rule levels, and the (J, J) job adjacency A = Σ A_rule ⊗ (Kronecker blocks over seed, n)."""
    rules = parse_snakefile(text if text is not None else T.TRAINING_GRAPH.read_text(encoding="utf-8"))
    A_rule, axes = rule_graph(rules)
    sizes = {"seed": len(cfg.seeds), "n": len(cfg.sizes)}
    A, off = dag.job_adjacency(A_rule, axes, sizes)
    return {"rules": rules, "names": [r["name"] for r in rules], "A_rule": A_rule, "axes": axes, "sizes": sizes,
            "levels": dag.levels(A_rule), "A": A, "off": off, "job_levels": dag.levels(A)}  # fmt: skip


def schedule(d: dict, seconds: dict) -> dict:
    """Max-plus finish times on the rule graph with the measured rule durations: with one worker per rule, the
    makespan is the critical path; one worker runs the sum."""
    dur = torch.tensor([seconds[n] for n in d["names"]], dtype=torch.float64)
    t = dag.finish_times(d["A_rule"].double(), dur)
    path = dag.critical_path(d["A_rule"].double(), t, dur)
    return {"finish": t, "duration": dur, "path": [d["names"][i] for i in path], "makespan": float(t.max()),
            "serial": float(dur.sum())}  # fmt: skip


# ---------------------------------------------------------------- training helpers
def clip_per_model(params: list[torch.Tensor], max_norm: float) -> None:
    """Scale model m's gradient to norm ≤ max_norm, model by model (a global clip would couple the models)."""
    params = [p for p in params if p.grad is not None]  # e.g. the CNN's digit head is unused in the stitched model
    sq = sum(p.grad.pow(2).flatten(1).sum(1) for p in params)
    scale = (max_norm / (sq.sqrt() + 1e-6)).clamp(max=1.0)
    for p in params:
        p.grad.mul_(scale.view(-1, *[1] * (p.dim() - 1)))


def fit(params: list[torch.Tensor], loss_fn, n_steps: int, lr: float, label: str) -> torch.Tensor:
    """AdamW (elementwise, so models stay independent), warm-up + cosine decay, per-model clipping.
    loss_fn(step) → (M,) losses. Returns the loss history (steps, M)."""
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    warm = max(1, min(30, n_steps // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: min(1.0, (t + 1) / warm) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * t / max(1, n_steps))))
    )
    hist = []
    for t in progress_steps(n_steps, label):
        loss = loss_fn(t)
        opt.zero_grad()
        loss.sum().backward()  # model m's loss reaches only model m's weights
        clip_per_model(params, 1.0)
        opt.step()
        sched.step()
        hist.append(loss.detach().cpu())
    return torch.stack(hist) if hist else torch.zeros(0, 1)


def joint_batch(data: dict, img_idx: torch.Tensor, op: torch.Tensor, b: torch.Tensor, train: bool = True) -> dict:
    """Joint examples: image idx (M, B) with prompt op, b → images, LM inputs, targets and masks (M, B, …).
    The operand word's letters are in `tokens` but sit under the soft prefix (positions < K) and are never targets."""
    x = (data["x_train"] if train else data["x_test"])[img_idx]
    a = (data["y_train"] if train else data["y_test"])[img_idx]
    tab = data["table"]
    tok = tab["tokens"][a, op, b]
    j = torch.arange(T.CTX - 1, device=tok.device)
    return {"x": x, "inp": tok[..., :-1], "tgt": tok[..., 1:], "mask": tab["text_mask"][a, op, b] & (j >= T.K - 1),
            "ans": tab["answer_mask"][a, op, b], "a": a, "op": op, "b": b}  # fmt: skip


def joint_draw(store: dict, rows: torch.Tensor) -> dict:
    """Rows (M, B) of each model's own joint set → a joint batch."""
    pool, si = store["pool"], store["seed_of"]
    return joint_batch(store["data"], pool["img"][si[:, None], rows], pool["op"][si[:, None], rows],
                       pool["b"][si[:, None], rows])  # fmt: skip


def task_loss(z: torch.Tensor, bt: dict) -> torch.Tensor:
    """Mean −log p over the text targets plus the same over the answer letters (the hard part).  → (M,)"""
    return eq.masked_ce(z, bt["tgt"], bt["mask"]) + eq.masked_ce(z, bt["tgt"], bt["ans"])


def joint_loss(store: dict, rows: torch.Tensor, w_cnn, w_con, w_lm) -> torch.Tensor:
    bt = joint_draw(store, rows)
    return task_loss(eq.stitched_logits(bt["x"], bt["inp"], w_cnn, w_con, w_lm), bt)


def moe_forward(bt: dict, w_cnn, w_con, w_lm, w_router, blank, soft: bool = False):
    """LM logits through the MoE input layer with a learned router (or the fixed one when w_router is None)."""
    ids, is_img = eq.position_items(bt["inp"])
    m, n, _ = ids.shape
    if w_router is None:
        G = eq.route_fixed(m, n, is_img)
        p = G
    else:
        G, p = eq.route_learned(eq.router_features(bt["x"], ids, is_img, blank), w_router, soft)
    return eq.lm_logits(eq.moe_inputs(bt["x"], ids, G, w_cnn, w_con, w_lm, blank), w_lm), G, p


# ---------------------------------------------------------------- the rules: one batched call each
def rule_train_cnn(st: dict) -> dict:
    cfg, data, S = st["cfg"], st["data"], len(st["cfg"].seeds)
    w = T.init_cnn(cfg.seeds, 1, st["dev"])
    rows = T.draws(cfg.seeds, [len(data["y_train"])] * S, cfg.steps_cnn, cfg.batch, 1, st["dev"])

    def loss(t):
        logits = eq.cnn_logits(eq.cnn_features(data["x_train"][rows[t]], w), w)
        return torch.nn.functional.cross_entropy(
            logits.transpose(1, 2), data["y_train"][rows[t]], reduction="none"
        ).mean(1)

    return {"w": w, "loss": fit(T.leaves(w), loss, cfg.steps_cnn, cfg.lr, "train_cnn")}


def rule_train_lm(st: dict) -> dict:
    cfg, tab, S = st["cfg"], st["data"]["table"], len(st["cfg"].seeds)
    w = T.init_lm(cfg.seeds, 2, st["dev"], cfg.d, cfg.heads, cfg.layers)
    tok, msk = tab["tokens"].reshape(300, T.CTX), tab["text_mask"].reshape(300, T.CTX - 1)
    ans = tab["answer_mask"].reshape(300, T.CTX - 1)
    rows = T.draws(cfg.seeds, [300] * S, cfg.steps_lm, cfg.batch, 2, st["dev"])

    def loss(t):
        x = tok[rows[t]]
        return task_loss(eq.lm_logits(eq.embed(x[..., :-1], w), w), {"tgt": x[..., 1:], "mask": msk[rows[t]],
                                                                    "ans": ans[rows[t]]})  # fmt: skip

    return {"w": w, "loss": fit(T.leaves(w), loss, cfg.steps_lm, cfg.lr_lm, "train_lm")}


def rule_train_monolith(st: dict) -> dict:
    """The non-DAG baseline: the same CNN + connector + LM, random init, trained end to end on joint data only."""
    cfg, sm, dev = st["cfg"], st["seeds_mn"], st["dev"]
    w_cnn, w_con = T.init_cnn(sm, 11, dev), T.init_connector(sm, 13, dev, cfg.d)
    w_lm = T.init_lm(sm, 12, dev, cfg.d, cfg.heads, cfg.layers)
    rows = T.draws(sm, st["n_mn"], cfg.steps_mono, cfg.batch, 5, dev)
    params = T.leaves(w_cnn) + T.leaves(w_con) + T.leaves(w_lm)
    hist = fit(params, lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm), cfg.steps_mono, cfg.lr_mono,
               "train_monolith")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "loss": hist}


def rule_train_connector(st: dict) -> dict:
    """Freeze the pretrained CNN and LM (broadcast seed → (seed, n)); only the connector learns."""
    cfg, sm, dev, N = st["cfg"], st["seeds_mn"], st["dev"], len(st["cfg"].sizes)
    w_cnn = T.clone(T.expand(st["train_cnn"]["w"], N), False)
    w_lm = T.clone(T.expand(st["train_lm"]["w"], N), False)
    w_con = T.init_connector(sm, 3, dev, cfg.d)
    init = T.clone(w_con, False)
    rows = T.draws(sm, st["n_mn"], cfg.steps_con, cfg.batch, 3, dev)
    hist = fit(T.leaves(w_con), lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm), cfg.steps_con, cfg.lr_con,
               "train_connector")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "con_init": init, "loss": hist}


def rule_global_finetune(st: dict) -> dict:
    """Unfreeze everything and train end to end through the DAG edge, gently."""
    cfg, sm, dev, c = st["cfg"], st["seeds_mn"], st["dev"], st["train_connector"]
    w_cnn, w_con, w_lm = (T.clone(c[k], True) for k in ("cnn", "con", "lm"))
    rows = T.draws(sm, st["n_mn"], cfg.steps_ft, cfg.batch, 4, dev)
    params = T.leaves(w_cnn) + T.leaves(w_con) + T.leaves(w_lm)
    hist = fit(params, lambda t: joint_loss(st, rows[t], w_cnn, w_con, w_lm), cfg.steps_ft, cfg.lr_ft,
               "global_finetune")  # fmt: skip
    return {"cnn": w_cnn, "con": w_con, "lm": w_lm, "loss": hist}


def rule_train_router(st: dict, soft: bool = False) -> dict:
    """Replace the fixed modality routing by a learned router (experts frozen): does it find it again?
    Switch top-1 (default): only the chosen expert runs, its output scaled by p. soft: train with the dense mixture
    G = p (both experts everywhere, so the gradient compares them), evaluate top-1 like the Switch router."""
    cfg, sm, dev, c = st["cfg"], st["seeds_mn"], st["dev"], st["train_connector"]
    w_r = T.init_router(sm, 4, dev)
    rows = T.draws(sm, st["n_mn"], cfg.steps_router, cfg.batch, 7 if soft else 6, dev)
    blank = st["data"]["blank"]

    def loss(t):
        bt = joint_draw(st, rows[t])
        logits, G, p = moe_forward(bt, c["cnn"], c["con"], c["lm"], w_r, blank, soft)
        return task_loss(logits, bt) + cfg.alpha * eq.balance_loss(G, p)

    name = "train_router_soft" if soft else "train_router"
    return {"router": w_r, "loss": fit(T.leaves(w_r), loss, cfg.steps_router, cfg.lr, name)}


@torch.no_grad()
def rule_evaluate(st: dict) -> dict:
    """Every stage on the same 1000 joint test examples, plus the unimodal checks."""
    cfg, data, dev = st["cfg"], st["data"], st["dev"]
    S, N, M = len(cfg.seeds), len(cfg.sizes), len(st["seeds_mn"])
    n_test = min(cfg.n_test, len(data["y_test"]))
    test = T.joint_test(n_test, dev)
    con, ft, mono = st["train_connector"], st["global_finetune"], st["train_monolith"]
    blank = data["blank"]
    routers = {"router": st["train_router"]["router"], "router_soft": st["train_router_soft"]["router"]}
    acc = {k: torch.zeros(M) for k in ("connector_init", "connector", "finetune", "router", "router_soft", "monolith")}
    route_acc = {k: torch.zeros(M) for k in routers}
    symbolic, cnn_acc = torch.zeros(S), torch.zeros(S)
    w_cnn1, w_lm1 = st["train_cnn"]["w"], st["train_lm"]["w"]
    for lo in range(0, n_test, 250):  # chunks of the test set
        sl = torch.arange(lo, min(lo + 250, n_test), device=dev)
        frac = len(sl) / n_test
        bt = joint_batch(data, sl[None].expand(M, -1), test["op"][sl][None].expand(M, -1),
                         test["b"][sl][None].expand(M, -1), train=False)  # fmt: skip

        def score(w_cnn, w_con, w_lm, bt=bt):
            return eq.exact(eq.stitched_logits(bt["x"], bt["inp"], w_cnn, w_con, w_lm), bt["tgt"], bt["ans"])

        acc["connector_init"] += frac * score(con["cnn"], con["con_init"], con["lm"]).float().mean(1).cpu()
        acc["connector"] += frac * score(con["cnn"], con["con"], con["lm"]).float().mean(1).cpu()
        acc["finetune"] += frac * score(ft["cnn"], ft["con"], ft["lm"]).float().mean(1).cpu()
        acc["monolith"] += frac * score(mono["cnn"], mono["con"], mono["lm"]).float().mean(1).cpu()
        is_img = torch.arange(T.CTX - 1, device=dev) < T.K
        for k, w_r in routers.items():  # both evaluated top-1
            logits, G, _ = moe_forward(bt, con["cnn"], con["con"], con["lm"], w_r, blank)
            acc[k] += frac * eq.exact(logits, bt["tgt"], bt["ans"]).float().mean(1).cpu()
            route_acc[k] += frac * ((G[..., 0] > 0) == is_img).float().mean((1, 2)).cpu()
        # unimodal: the CNN alone, and the symbolic pipeline (CNN's argmax digit written as a word, then the LM)
        x1 = data["x_test"][sl][None].expand(S, -1, -1, -1)
        pred = eq.cnn_logits(eq.cnn_features(x1, w_cnn1), w_cnn1).argmax(-1)  # (S, n)
        cnn_acc += frac * (pred == data["y_test"][sl]).float().mean(1).cpu()
        tok = data["table"]["tokens"][pred, test["op"][sl], test["b"][sl]]
        truth = data["table"]["tokens"][data["y_test"][sl], test["op"][sl], test["b"][sl]]
        ans = data["table"]["answer_mask"][data["y_test"][sl], test["op"][sl], test["b"][sl]]
        z = eq.lm_logits(eq.embed(tok[..., :-1], w_lm1), w_lm1)
        symbolic += frac * eq.exact(z, truth[..., 1:].expand(S, -1, -1), ans.expand(S, -1, -1)).float().mean(1).cpu()
    # the LM alone on every text-only prompt of the language
    tok = data["table"]["tokens"].reshape(300, T.CTX)
    ans = data["table"]["answer_mask"].reshape(300, T.CTX - 1)
    z = eq.lm_logits(eq.embed(tok[None, :, :-1].expand(S, -1, -1), w_lm1), w_lm1)
    lm_acc = eq.exact(z, tok[None, :, 1:].expand(S, -1, -1), ans[None].expand(S, -1, -1)).float().mean(1).cpu()
    shape = (S, N)
    return {"acc": {k: v.reshape(shape) for k, v in acc.items()},
            "route_acc": {k: v.reshape(shape) for k, v in route_acc.items()},
            "cnn_acc": cnn_acc, "lm_acc": lm_acc, "symbolic": symbolic, "test": test}  # fmt: skip


RULES = {"train_cnn": rule_train_cnn, "train_lm": rule_train_lm, "train_monolith": rule_train_monolith,
         "train_connector": rule_train_connector, "global_finetune": rule_global_finetune,
         "train_router": rule_train_router, "train_router_soft": lambda st: rule_train_router(st, soft=True),
         "evaluate": rule_evaluate}  # fmt: skip


def make_store(cfg: Config, dev: torch.device) -> dict:
    """Data and the model-axis bookkeeping every rule reads. Model m = (seed, n), seed-major (as the job DAG)."""
    data = T.load_mnist(dev)
    data["table"] = {k: v.to(dev) for k, v in T.sentence_table().items()}
    S, N = len(cfg.seeds), len(cfg.sizes)
    return {"cfg": cfg, "dev": dev, "data": data, "pool": T.joint_pool(cfg.seeds, len(data["y_train"]), dev),
            "seeds_mn": [s for s in cfg.seeds for _ in cfg.sizes], "n_mn": [n for _ in cfg.seeds for n in cfg.sizes],
            "seed_of": torch.arange(S, device=dev).repeat_interleave(N)}  # fmt: skip


def run_engine(cfg: Config, dev: torch.device, d: dict | None = None) -> tuple[dict, dict, dict]:
    """Run the training DAG: rules level by level (levels of A_rule), each rule one batched call over its models.
    Returns the store, the measured seconds per rule and the DAG."""
    d = d or build_dag(cfg)
    st = make_store(cfg, dev)
    seconds = {}
    for k in range(int(d["levels"].max()) + 1):
        for r in (d["levels"] == k).nonzero().flatten().tolist():
            name = d["names"][r]
            t0 = time.perf_counter()
            st[name] = RULES[name](st)
            seconds[name] = time.perf_counter() - t0
    return st, seconds, d


@torch.no_grad()
def greedy(w_cnn, w_con, w_lm, x: torch.Tensor, tokens: torch.Tensor) -> list[str]:
    """Let ONE stitched model (stack of 1) write the result word after '… is ' letter by letter.  x (N, 28, 28)."""
    tok = tokens.clone()[None]
    start = [T.decode(r).index("=") + 1 for r in tokens]
    pos = torch.tensor(start, device=x.device)
    rows = torch.arange(len(start), device=x.device)
    for k in range(6):  # 'seven.' is the longest answer
        z = eq.stitched_logits(x[None], tok[:, :, :-1], w_cnn, w_con, w_lm)[0]
        p = (pos + k).clamp(max=T.CTX - 1)
        nxt = z[rows, p - 1].argmax(-1)
        tok[0, rows, p] = nxt
    out = []
    for r, s in zip(tok[0], start):
        text = T.decode(r[s:])
        out.append(re.split(r"[._]", text)[0])
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/stitch_llm_cnn_dag/plot.py
"""Animate the experiment: the training DAG runs level by level while the per-stage loss curves grow; then the
accuracy of every stage against the monolith (seed dots), the sample-efficiency curves, the router G (fixed vs
learned) and test examples. A static PNG holds the measured schedule (serial vs critical path) and G.
Tables: stages.csv (accuracy per stage, seed and joint-set size) and timings.csv (seconds per rule)."""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from problems._common.plotting import ACCENT, export_plot_data, save_png
from problems._common.progress import report

from . import tensors as T

STAGE = {  # key: (label, colour)
    "connector_init": ("untrained conn.", "#9ca3af"),
    "connector": ("L1 connector", ACCENT[0]),
    "finetune": ("L2 fine-tune", ACCENT[2]),
    "router": ("Switch router", ACCENT[3]),
    "router_soft": ("dense router", ACCENT[6]),
    "monolith": ("monolith", ACCENT[1]),
}
RULE_COLOUR = {"train_cnn": ACCENT[5], "train_lm": ACCENT[4], "train_monolith": ACCENT[1],
               "train_connector": ACCENT[0], "global_finetune": ACCENT[2], "train_router": ACCENT[3],
               "train_router_soft": ACCENT[6], "evaluate": "#374151"}  # fmt: skip
SHORT = {"train_cnn": "CNN", "train_lm": "LM", "train_monolith": "monolith",
         "train_connector": "connector", "global_finetune": "fine-tune", "train_router": "Switch",
         "train_router_soft": "dense", "evaluate": "eval"}  # fmt: skip


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 6) -> Path:
    """Render every frame, then quantise all of them with ONE palette so colours never drift between frames."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)  # numbers behind the final frame (Results page)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def _node_xy(res: dict) -> dict:
    """Rule → (x, y): x = level, rules of one level stacked."""
    xy, lev = {}, res["levels"]
    for k in range(max(lev) + 1):
        group = [n for n, lv in zip(res["names"], lev) if lv == k]
        for i, n in enumerate(group):
            xy[n] = (k, (len(group) - 1) / 2 - i)
    return xy


def _draw_dag(ax, res: dict, done_level: float) -> None:
    ax.clear()
    xy = _node_xy(res)
    path = set(zip(res["path"][:-1], res["path"][1:]))
    A = res["A_rule"]
    for s, rs in enumerate(res["names"]):
        for r, rr in enumerate(res["names"]):
            if A[s][r]:
                crit = (rr, rs) in path
                (x0, y0), (x1, y1) = xy[rr], xy[rs]
                ax.annotate("", (x1 - 0.32, y1), (x0 + 0.32, y0),
                            arrowprops=dict(arrowstyle="-|>", lw=2.2 if crit else 1.0,
                                            color="#dc2626" if crit else "#9ca3af"))  # fmt: skip
    for n, lv in zip(res["names"], res["levels"]):
        x, y = xy[n]
        on = lv < done_level
        ax.text(x, y, f"{SHORT[n]}\n{res['seconds'][n]:.1f} s" if on else SHORT[n], ha="center", va="center",
                fontsize=9, color="white" if on else "#374151",
                bbox=dict(boxstyle="round,pad=0.3", fc=RULE_COLOUR[n] if on else "white", ec=RULE_COLOUR[n],
                          lw=1.5))  # fmt: skip
    ax.set_xlim(-0.55, max(res["levels"]) + 0.55)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xticks(range(max(res["levels"]) + 1))
    ax.set_yticks([])
    ax.set_xlabel("level $\\ell$ (dimensionless)", fontsize=10)
    ax.tick_params(labelsize=10)
    ax.set_title(f"training DAG: critical path {res['makespan']:.0f} s, serial {res['serial']:.0f} s", fontsize=11)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)


def animate(res: dict, path: Path, n_frames: int = 36, hold: int = 10) -> Path:
    fig = plt.figure(figsize=(14, 8.2))
    gs = fig.add_gridspec(2, 4, width_ratios=[1.45, 1.1, 1.3, 1.0], hspace=0.75, wspace=0.42,
                          left=0.04, right=0.985, top=0.93, bottom=0.10)  # fmt: skip
    ax_dag, ax_loss = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    ax_bar, ax_eff = fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[0, 3])
    g_axes = [fig.add_subplot(a) for a in gs[1, 0:2].subgridspec(1, 3, wspace=0.12)]
    ex_axes = [fig.add_subplot(gs[1, 2 + i // 2].subgridspec(2, 1, hspace=0.9)[i % 2]) for i in range(4)]
    sizes = np.array(res["sizes"])
    S = len(res["seeds"])
    n_levels = max(res["levels"]) + 1
    loss = {k: v.numpy() for k, v in res["loss"].items()}
    lmax = max(float(np.nanmax(v)) for v in loss.values() if v.size)
    lmin = min(float(np.nanmin(v)) for v in loss.values() if v.size)
    rng = np.random.default_rng(0)
    jit = rng.uniform(-0.12, 0.12, S)

    # ---- static panels: G fixed vs learned
    ra = res["route_acc_show"]
    panels = ((res["G_fixed"], "fixed $G$\n(by data type)"),
              (res["G_learned"]["router"], f"Switch top-1\nroute acc. {ra['router']:.2f}"),
              (res["G_learned"]["router_soft"], f"dense-trained\nroute acc. {ra['router_soft']:.2f}"))  # fmt: skip
    for i, (ax, (G, title)) in enumerate(zip(g_axes, panels)):
        im = ax.imshow(G, cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
        ax.axvline(T.K - 0.5, color="#dc2626", lw=1.5, ls="--")
        ax.set_xlabel("position $t$ (dimensionless)", fontsize=10)
        if i == 0:
            ax.set_ylabel("test example (dimensionless)", fontsize=10)
        else:
            ax.set_yticklabels([])
        ax.set_title(title, fontsize=11)
        ax.tick_params(labelsize=10)
    cb = fig.colorbar(im, ax=g_axes, fraction=0.04, pad=0.02)
    cb.set_label("$p_{t,\\mathrm{img}}$, image expert (dimensionless)", fontsize=10)
    cb.ax.tick_params(labelsize=9)

    def update(f: int) -> None:
        prog = min(1.0, f / max(1, n_frames - 1))  # fraction of the training run shown
        _draw_dag(ax_dag, res, prog * n_levels if f < n_frames else n_levels + 1)
        ax_loss.clear()
        for rule, v in loss.items():
            if not v.size:
                continue
            k = max(1, int(round(prog * len(v))))
            steps = np.arange(1, len(v) + 1)
            for m in range(v.shape[1]):
                ax_loss.plot(steps[:k], v[:k, m], color=RULE_COLOUR[rule], lw=0.6, alpha=0.35)
            ax_loss.plot(steps[:k], v[:k].mean(1), color=RULE_COLOUR[rule], lw=1.8, label=SHORT[rule])
        ax_loss.set_yscale("log")
        ax_loss.set_ylim(lmin * 0.8, lmax * 1.3)
        ax_loss.set_xlim(1, max(len(v) for v in loss.values()))
        ax_loss.set_xlabel("optimiser step (dimensionless)", fontsize=10)
        ax_loss.set_ylabel("loss $\\mathcal{L}$ (nats)", fontsize=10)
        ax_loss.set_title("per-stage loss, every model", fontsize=11)
        ax_loss.legend(fontsize=9, loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.2), frameon=False,
                       handlelength=1.2, columnspacing=0.8)  # fmt: skip
        ax_loss.tick_params(labelsize=10)

        show = f >= n_frames
        ax_bar.clear()
        keys = list(STAGE)
        big = len(sizes) - 1
        for i, k in enumerate(keys):
            a = res["acc"][k][:, big].numpy()
            ax_bar.bar(i, a.mean() if show else 0, color=STAGE[k][1], width=0.7)
            if show:
                ax_bar.scatter(i + jit, a, s=18, color="black", zorder=3)
        if show:
            sym = res["symbolic"].numpy()
            ax_bar.axhline(sym.mean(), color="#6b7280", lw=1.2, ls=":", label="CNN label → text → LM")
            ax_bar.axhline(0.1, color="#9ca3af", lw=1.0, ls="--", label="chance")
            ax_bar.legend(fontsize=9, loc="upper left", framealpha=0.9)
        ax_bar.set_xticks(range(len(keys)), [STAGE[k][0] for k in keys], fontsize=9, rotation=35, ha="right",
                          rotation_mode="anchor")  # fmt: skip
        ax_bar.set_ylim(0, 1.18)
        ax_bar.set_xlim(-0.6, len(keys) - 0.4)
        ax_bar.set_ylabel("test accuracy (dimensionless)", fontsize=10)
        ax_bar.set_title(f"image + prompt → answer, $n$ = {sizes[big]}", fontsize=11)
        ax_bar.tick_params(axis="y", labelsize=10)

        ax_eff.clear()
        for k in ("connector", "finetune", "monolith"):
            a = res["acc"][k].numpy()  # (S, N)
            if show:
                for s in range(S):
                    ax_eff.scatter(sizes * (1 + 0.04 * jit[s]), a[s], s=14, color=STAGE[k][1], alpha=0.7)
                ax_eff.plot(
                    sizes, a.mean(0), "-o", color=STAGE[k][1], lw=1.8, ms=4, label=STAGE[k][0].replace("\n", " ")
                )
        ax_eff.axhline(0.1, color="#9ca3af", lw=1.0, ls="--")
        ax_eff.set_xscale("log")
        ax_eff.set_xlim(sizes.min() / 1.5, sizes.max() * 1.5)
        ax_eff.set_ylim(0, 1.05)
        ax_eff.set_xlabel("joint examples $n$ (dimensionless)", fontsize=10)
        ax_eff.set_ylabel("test accuracy (dimensionless)", fontsize=10)
        ax_eff.set_title("sample efficiency", fontsize=11)
        if show:
            ax_eff.legend(fontsize=9, loc="right", bbox_to_anchor=(1.0, 0.4), framealpha=0.9)
        ax_eff.tick_params(labelsize=10)

        for ax, ex in zip(ex_axes, res["examples"]):
            ax.clear()
            ax.set_xticks([])
            ax.set_yticks([])
            ax.imshow(ex["img"], cmap="gray_r", extent=(0, 1, 0, 1))
            ax.set_xlim(0, 4.6)
            ax.set_ylim(0, 1)
            ax.set_aspect("equal")
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.text(1.12, 0.78, f"[img]{ex['prompt']}", fontsize=10, va="center", family="monospace")
            if show:
                ok_d, ok_m = ex["dag"] == ex["truth"], ex["mono"] == ex["truth"]
                ax.text(1.12, 0.42, f"DAG: {ex['dag']}", fontsize=10, va="center", family="monospace",
                        color=ACCENT[2] if ok_d else ACCENT[1])  # fmt: skip
                ax.text(1.12, 0.08, f"mono: {ex['mono']}", fontsize=10, va="center", family="monospace",
                        color=ACCENT[2] if ok_m else ACCENT[1])  # fmt: skip
        fig.suptitle("Separately trained CNN + LM, stitched by one DAG edge vs a monolith trained end to end",
                     fontsize=13)  # fmt: skip

    return save_gif_shared_palette(fig, update, list(range(n_frames + hold)), path)


def schedule_png(res: dict, path: Path) -> Path:
    """Measured rule times as a Gantt chart: top, one worker running the rules level by level (serial); below, one
    worker per rule, each starting when its inputs are done (max-plus finish times); black edges = critical path."""
    names = res["names"]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    t0 = 0.0
    top = len(names) + 0.6
    for i, n in enumerate(names):
        d, fin = res["seconds"][n], res["finish"][i]
        ax.barh(top, d, left=t0, height=0.7, color=RULE_COLOUR[n], edgecolor="white")
        crit = n in res["path"]
        ax.barh(len(names) - 1 - i, d, left=fin - d, height=0.6, color=RULE_COLOUR[n],
                edgecolor="#111827" if crit else "white", linewidth=2.0 if crit else 0.5)  # fmt: skip
        t0 += d
    labels = [SHORT[n] + (" (CP)" if n in res["path"] else "") for n in names][::-1] + ["serial"]
    ax.set_yticks(list(range(len(names))) + [top], labels, fontsize=10)
    ax.axhline(len(names) - 0.2, color="#9ca3af", lw=1, ls="--")
    ax.set_xlabel("wall time $t$ (s)", fontsize=10)
    ax.set_xlim(0, res["serial"] * 1.02)
    ax.set_ylim(-0.6, top + 0.6)
    ax.tick_params(labelsize=10)
    ax.set_title(f"measured schedule: serial {res['serial']:.0f} s; one worker per rule {res['makespan']:.0f} s "
                 "(critical path, CP)", fontsize=11)  # fmt: skip
    fig.tight_layout()
    return save_png(fig, path)


def write_tables(res: dict, out_dir: Path) -> None:
    with open(out_dir / "stages.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["stage", "seed", "n_joint", "test_accuracy"])
        for k in STAGE:
            for si, s in enumerate(res["seeds"]):
                for ni, n in enumerate(res["sizes"]):
                    w.writerow([k, s, n, f"{float(res['acc'][k][si, ni]):.4f}"])
        for si, s in enumerate(res["seeds"]):
            for k, ra in res["route_acc"].items():
                for ni, n in enumerate(res["sizes"]):
                    w.writerow([f"{k}_routing_accuracy", s, n, f"{float(ra[si, ni]):.4f}"])
            w.writerow(["cnn_alone_digit_accuracy", s, "", f"{float(res['cnn_acc'][si]):.4f}"])
            w.writerow(["lm_alone_text_prompts", s, "", f"{float(res['lm_acc'][si]):.4f}"])
            w.writerow(["symbolic_cnn_label_to_lm", s, "", f"{float(res['symbolic'][si]):.4f}"])
    with open(out_dir / "timings.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["rule", "level", "seconds", "finish_one_worker_per_rule_s", "on_critical_path", "models"])
        for i, n in enumerate(res["names"]):
            w.writerow([n, res["levels"][i], f"{res['seconds'][n]:.2f}", f"{res['finish'][i]:.2f}",
                        int(n in res["path"]), res["models"][n]])  # fmt: skip


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'seeds': 2,
    'steps_cnn': 150,
    'steps_lm': 600,
    'steps_con': 150,
    'steps_ft': 80,
    'steps_router': 60,
    'lr_ft': 0.0003,
    'alpha': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.stitch_llm_cnn_dag")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.stitch_llm_cnn_dag.main")
problem_dir = Path("problems/ai/stitch_llm_cnn_dag")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
